# Incremental Meta-Learning Pipeline

This notebook implements **incremental meta-learning during training**:

```
Training Sample 1-50          Training Sample 51-100         Training Sample 101-150
        │                             │                              │
        ▼                             ▼                              ▼
┌──────────────┐              ┌──────────────┐               ┌──────────────┐
│ Run with     │              │ Run with     │               │ Run with     │
│ Base Prompt  │              │ Prompt v1    │               │ Prompt v2    │
└──────────────┘              └──────────────┘               └──────────────┘
        │                             │                              │
        ▼                             ▼                              ▼
┌──────────────┐              ┌──────────────┐               ┌──────────────┐
│ Errors 1-50  │              │ Errors 51-100│               │ Errors 101-150│
└──────────────┘              └──────────────┘               └──────────────┘
        │                             │                              │
        ▼                             ▼                              ▼
┌──────────────┐              ┌──────────────┐               ┌──────────────┐
│ Analyze &    │──────────────│ Analyze &    │───────────────│ Analyze &    │
│ Generate     │              │ Generate     │               │ Generate     │
│ Prompt v1    │              │ Prompt v2    │               │ Prompt v3    │
└──────────────┘              └──────────────┘               └──────────────┘
```

At each checkpoint:
1. Read `debug_failures.txt` (current errors)
2. Find NEW errors since last checkpoint
3. Analyze errors with LLM
4. Generate enhanced prompt building on previous learnings
5. Continue training with improved prompt

The final enhanced prompt accumulates all learnings for the TEST phase.

## 1. Setup and Imports

In [1]:
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

# Add this folder and lib/ to path (so imports like neurosymbolic_pipeline work)
_nb_dir = Path.cwd() / "pamaldi_semeval_2026_11_task1" if (Path.cwd() / "pamaldi_semeval_2026_11_task1").exists() else Path.cwd()
sys.path.insert(0, str(_nb_dir))
import setup_path

# Load AWS credentials (this folder or parent)
from load_credentials import load_credentials_from_file
_creds = _nb_dir / "aws_credentials.txt" if (_nb_dir / "aws_credentials.txt").exists() else _nb_dir.parent / "aws_credentials.txt"
load_credentials_from_file(str(_creds))
print('✓ Credentials loaded')

# Imports
from bedrock_client_bearer import BedrockClientBearer as BedrockClient
from neurosymbolic_pipeline import NeuroSymbolicPipeline
from syllogism_extractor import SyllogismExtractor
from extraction_reflexion import ExtractionReflexion
from incremental_meta_learning import IncrementalMetaLearning
from evaluation import NeuroSymbolicEvaluator

print('✓ Imports successful')

✓ Credentials loaded
✓ Imports successful


## 2. Configuration

In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

# Model
MODEL_ID = "qwen.qwen3-32b-v1:0"

# Number of samples
NUM_TRAIN = 200    # Training samples (-1 for all 1000)
NUM_TEST = -1     # Test samples (-1 for all 191)

# ============================================================
# INCREMENTAL META-LEARNING CONFIGURATION
# ============================================================
CHECKPOINT_INTERVAL = 50   # Create checkpoint every N samples
MAX_FEW_SHOTS = 10         # Max few-shot examples to keep
MAX_GUIDELINES = 20        # Max guidelines to keep

# Pipeline configuration
PIPELINE_CONFIG = {
    "use_reflexion": True,
    "max_reflexion_attempts": 3,
    "use_self_consistency": True,
    "num_consistency_samples": 4,
    "temperature_schedule": [0.0, 0.3, 0.5, 0.7],
    "verify_figure": True,
    "use_fallback": True
}

# Data paths
TRAIN_DATA_PATH = "data/train_data.json"
TEST_DATA_PATH = "data/test_data_subtask_1.json"
BASE_PROMPT_PATH = "prompts/structure_extraction.txt"
RESULTS_DIR = "../semeval_results"

# ============================================================
print(f"Model: {MODEL_ID}")
print(f"NUM_TRAIN: {NUM_TRAIN} {'(all)' if NUM_TRAIN == -1 else ''}")
print(f"NUM_TEST: {NUM_TEST} {'(all)' if NUM_TEST == -1 else ''}")
print(f"\nIncremental Meta-Learning:")
print(f"  Checkpoint interval: every {CHECKPOINT_INTERVAL} samples")
print(f"  Max few-shots: {MAX_FEW_SHOTS}")
print(f"  Max guidelines: {MAX_GUIDELINES}")

Model: qwen.qwen3-32b-v1:0
NUM_TRAIN: 200 
NUM_TEST: -1 (all)

Incremental Meta-Learning:
  Checkpoint interval: every 50 samples
  Max few-shots: 10
  Max guidelines: 20


## 3. Initialize Client

In [3]:
# Initialize Bedrock client
client = BedrockClient(model_id=MODEL_ID)
evaluator = NeuroSymbolicEvaluator()

print("✓ Client initialized")

✓ Client initialized


## 4. Load Data

In [4]:
# Load training data
with open(TRAIN_DATA_PATH, 'r', encoding='utf-8') as f:
    train_data = json.load(f)

if NUM_TRAIN > 0:
    train_data = train_data[:NUM_TRAIN]

print(f"Loaded {len(train_data)} training examples")

# Load base prompt
with open(BASE_PROMPT_PATH, 'r', encoding='utf-8') as f:
    base_prompt = f.read()

print(f"Base prompt loaded: {len(base_prompt)} chars")

Loaded 200 training examples
Base prompt loaded: 15878 chars


## 5. TRAINING PHASE with Incremental Meta-Learning

This is the main training loop with checkpoints every N samples.

In [5]:
# Create output directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = os.path.join(RESULTS_DIR, f"incremental_train_{timestamp}")
os.makedirs(output_dir, exist_ok=True)

# Save base prompt as v0
os.makedirs(os.path.join(output_dir, "prompts"), exist_ok=True)
with open(os.path.join(output_dir, "prompts", "prompt_v0.txt"), 'w', encoding='utf-8') as f:
    f.write(base_prompt)

# Initialize meta-learning checkpoint manager
meta_learner = IncrementalMetaLearning(
    output_dir=output_dir,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    max_few_shots=MAX_FEW_SHOTS,
    max_guidelines=MAX_GUIDELINES,
    verbose=True
)

# Initialize pipeline with base prompt
current_prompt_path = BASE_PROMPT_PATH

pipeline = NeuroSymbolicPipeline(
    bedrock_client=client,
    prompt_path=current_prompt_path,
    results_dir=output_dir,
    run_name="train",
    **PIPELINE_CONFIG
)

# Debug failures file path
debug_failures_path = pipeline.debug_log_file

print(f"\n✓ Pipeline initialized")
print(f"  Output dir: {output_dir}")
print(f"  Debug failures: {debug_failures_path}")

INCREMENTAL META-LEARNING INITIALIZED
Checkpoint interval: every 50 samples
Output directory: ..\semeval_results\incremental_train_20260301_102956
Max few-shots: 10
Max guidelines: 20


  System prompt saved to: ../semeval_results\incremental_train_20260301_102956\neurosymbolic_train_20260301_102956\system_prompt.txt
  Results dir: ../semeval_results\incremental_train_20260301_102956\neurosymbolic_train_20260301_102956
✓ NeuroSymbolic Pipeline initialized
  Extractor: Original (LLM figure)
  Use reflexion: True
  Use self-consistency: True
    Samples: 4, Temperatures: [0.0, 0.3, 0.5, 0.7]
  Verify figure: True
  Use fallback: True
    Fallback self-consistency: False
  Results dir: ../semeval_results\incremental_train_20260301_102956\neurosymbolic_train_20260301_102956

✓ Pipeline initialized
  Output dir: ../semeval_results\incremental_train_20260301_102956
  Debug failures: ../semeval_results\incremental_train_20260301_102956\neurosymbolic_train_20260301_102956\debug_failures.txt


In [6]:
# Process training samples with checkpoints
results = []

print(f"\nStarting training with incremental meta-learning...")
print(f"Checkpoint interval: every {CHECKPOINT_INTERVAL} samples")
print("-" * 60)

for i, item in enumerate(tqdm(train_data, desc="Training")):
    # Get text
    text = item.get("syllogism") or item.get("text", "")
    syllogism_id = item.get("id")
    
    # Process single syllogism
    result = pipeline.process_syllogism(text, syllogism_id)
    
    # Add ground truth
    if "validity" in item:
        gt = item["validity"]
        if isinstance(gt, bool):
            result["ground_truth"] = "VALID" if gt else "INVALID"
        else:
            result["ground_truth"] = gt
    
    # Add plausibility
    if "plausibility" in item:
        plaus = item["plausibility"]
        if isinstance(plaus, bool):
            result["plausibility"] = "PLAUSIBLE" if plaus else "IMPLAUSIBLE"
        else:
            result["plausibility"] = plaus
    
    results.append(result)
    
    # Save individual log
    pipeline._save_log(result)
    
    # Log failures in real-time
    if result.get("ground_truth"):
        if result.get("method") == "fallback":
            if result["prediction"] == result["ground_truth"]:
                pipeline.pipeline_stats["fallback_correct"] += 1
            pipeline._log_failure(result, "FALLBACK_USED")
        elif not result["extraction_success"] and result["prediction"] is None:
            pipeline._log_failure(result, "EXTRACTION_ERROR")
        elif result["prediction"] != result["ground_truth"]:
            if result["prediction"] == "VALID":
                pipeline._log_failure(result, "MISMATCH_FALSE_VALID")
            else:
                pipeline._log_failure(result, "MISMATCH_FALSE_INVALID")
    
    # Check if we should checkpoint
    if meta_learner.should_checkpoint(i):
        print(f"\n\n{'='*60}")
        print(f"CHECKPOINT at sample {i+1}")
        print(f"{'='*60}")
        
        # Process checkpoint and get enhanced prompt
        new_prompt_path = meta_learner.process_checkpoint(
            sample_index=i,
            debug_failures_path=debug_failures_path,
            llm_client=client,
            base_prompt=base_prompt
        )
        
        # Reload extractor with enhanced prompt
        if new_prompt_path:
            print(f"\n→ Updating pipeline with enhanced prompt")
            
            # Create new extractor with enhanced prompt
            pipeline.extractor = SyllogismExtractor(
                client,
                prompt_path=new_prompt_path,
                use_self_consistency=PIPELINE_CONFIG.get("use_self_consistency", False),
                num_consistency_samples=PIPELINE_CONFIG.get("num_consistency_samples", 3),
                temperature_schedule=PIPELINE_CONFIG.get("temperature_schedule"),
                verify_figure=PIPELINE_CONFIG.get("verify_figure", False),
                verbose=pipeline.verbose
            )
            
            # Update reflexion if used
            if pipeline.use_reflexion:
                pipeline.reflexion = ExtractionReflexion(
                    client,
                    pipeline.extractor,
                    PIPELINE_CONFIG.get("max_reflexion_attempts", 3),
                    use_self_consistency=PIPELINE_CONFIG.get("use_self_consistency", False),
                    verify_figure=PIPELINE_CONFIG.get("verify_figure", False),
                    num_consistency_samples=PIPELINE_CONFIG.get("num_consistency_samples", 3),
                    temperature_schedule=PIPELINE_CONFIG.get("temperature_schedule"),
                    verbose=pipeline.verbose
                )
            
            current_prompt_path = new_prompt_path
            print(f"✓ Pipeline updated")
        
        print(f"{'='*60}\n")

# Generate final summary
pipeline._generate_summary(results)

print("\n✓ Training complete!")


Starting training with incremental meta-learning...
Checkpoint interval: every 50 samples
------------------------------------------------------------


Training:  24%|██▍       | 49/200 [10:07<39:19, 15.63s/it]================================================================================
CHECKPOINT 1 (after sample 50)
Timestamp: 2026-03-01T10:40:30.768039

--- STEP 1: Reading current errors ---
Total errors in debug_failures.txt: 3

--- STEP 2: Loading previous checkpoint ---
Previous step: 0
Previous cumulative learnings: 0 few-shots

--- STEP 3: Identifying NEW errors ---
New errors since last checkpoint: 3

--- STEP 4: Analyzing new errors with LLM ---
Analyzing 3 errors with LLM...




CHECKPOINT at sample 50


Analysis complete:
  - Patterns found: 1
  - Few-shots generated: 3
  - Warnings identified: 2
  - Guidelines added: 5

--- STEP 5: Merging with previous learnings ---
Merged learnings:
  - Total patterns: 1
  - Total few-shots: 3
  - Total warnings: 2
  - Total guidelines: 5

--- STEP 6: Generating enhanced prompt ---
Generating enhanced prompt...
Prompt enhanced:
  - Original length: 15878 chars
  - Enhanced length: 18358 chars
  - Added: 2480 chars
Enhanced prompt saved: ..\semeval_results\incremental_train_20260301_102956\prompts\prompt_v1.txt

--- STEP 7: Saving checkpoint ---

CHECKPOINT 1 COMPLETE (sample 50)
  New errors analyzed: 3
  Cumulative few-shots: 3
  Cumulative guidelines: 5
  New prompt: ..\semeval_results\incremental_train_20260301_102956\prompts\prompt_v1.txt
  Detailed log: ..\semeval_results\incremental_train_20260301_102956\logs\checkpoint_0050.log

CHECKPOINT COMPLETE
Training:  25%|██▌       | 50/200 [10:38<50:14, 20.09s/it]


→ Updating pipeline with enhanced prompt
✓ Pipeline updated



...


→ Updating pipeline with enhanced prompt
✓ Pipeline updated



Training: 100%|█████████▉| 199/200 [49:58<00:14, 14.53s/it]================================================================================
CHECKPOINT 4 (after sample 200)
Timestamp: 2026-03-01T11:20:07.782226

--- STEP 1: Reading current errors ---
Total errors in debug_failures.txt: 11

--- STEP 2: Loading previous checkpoint ---
Previous step: 3
Previous cumulative learnings: 8 few-shots

--- STEP 3: Identifying NEW errors ---
New errors since last checkpoint: 3

--- STEP 4: Analyzing new errors with LLM ---
Analyzing 3 errors with LLM...




CHECKPOINT at sample 200


Analysis complete:
  - Patterns found: 1
  - Few-shots generated: 3
  - Warnings identified: 3
  - Guidelines added: 6

--- STEP 5: Merging with previous learnings ---
Limiting few-shots from 11 to 10
Limiting guidelines from 21 to 20
Merged learnings:
  - Total patterns: 4
  - Total few-shots: 10
  - Total warnings: 10
  - Total guidelines: 20

--- STEP 6: Generating enhanced prompt ---
Generating enhanced prompt...
Prompt enhanced:
  - Original length: 15878 chars
  - Enhanced length: 24585 chars
  - Added: 8707 chars
Enhanced prompt saved: ..\semeval_results\incremental_train_20260301_102956\prompts\prompt_v4.txt

--- STEP 7: Saving checkpoint ---

CHECKPOINT 4 COMPLETE (sample 200)
  New errors analyzed: 3
  Cumulative few-shots: 10
  Cumulative guidelines: 20
  New prompt: ..\semeval_results\incremental_train_20260301_102956\prompts\prompt_v4.txt
  Detailed log: ..\semeval_results\incremental_train_20260301_102956\logs\checkpoint_0200.log

CHECKPOINT COMPLETE
Training: 100%|██████


→ Updating pipeline with enhanced prompt
✓ Pipeline updated


✓ Summary saved to: ../semeval_results\incremental_train_20260301_102956\neurosymbolic_train_20260301_102956
  Methods: 181 symbolic, 19 fallback, 0 failed

  ┌─────────────────────────────────────┐
  │  COMBINED SCORE:    55.95           │
  │  Accuracy:          93.00%          │
  │  Content Effect:     0.94           │
  └─────────────────────────────────────┘

  Accuracy breakdown: Symbolic 93.92%, Fallback 84.21%

  Failures logged: 14 (see debug_failures.txt)

✓ Training complete!


## 6. Finalize Meta-Learning

In [7]:
# Finalize meta-learning and get final prompt
final_prompt_path = meta_learner.finalize()

print("\n" + "=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"Total samples: {len(results)}")
print(f"Checkpoints created: {meta_learner.current_step}")
print(f"Total errors seen: {len(meta_learner.all_errors_seen)}")
if final_prompt_path:
    print(f"\nFinal enhanced prompt: {final_prompt_path}")
    print("\n→ Use this prompt for the TEST phase!")
print("=" * 60)


FINALIZING META-LEARNING
Final prompt: ..\semeval_results\incremental_train_20260301_102956\prompts\prompt_final.txt

META-LEARNING SUMMARY:
  - Total checkpoints: 4
  - Total samples processed: 200
  - Total errors seen: 11
  - Final few-shots: 10
  - Final guidelines: 20



TRAINING SUMMARY
Total samples: 200
Checkpoints created: 4
Total errors seen: 11

Final enhanced prompt: ..\semeval_results\incremental_train_20260301_102956\prompts\prompt_final.txt

→ Use this prompt for the TEST phase!


## 7. Evaluate Training Results

In [8]:
# Evaluate training results
train_metrics = evaluator.evaluate(results)
evaluator.print_report(train_metrics)


NEURO-SYMBOLIC EVALUATION REPORT

Total Items: 200
Processed: 200
Extraction Rate: 90.5%

Overall Accuracy: 93.00%
  Correct: 186
  Incorrect: 14

Content Effect: 1.44%
  Plausible Accuracy: 93.75% (n=96)
  Implausible Accuracy: 92.31% (n=104)

By Ground Truth:
  VALID accuracy: 92.78% (n=97)
  INVALID accuracy: 93.20% (n=103)

Error Breakdown:
  false_invalid: 7
  false_valid: 7

Most Common Form Errors:
  AAA-1: 2
  IAI-2: 2
  AOO-2: 2
  AAI-3: 2
  IAI-1: 1



## 7.1 Official Evaluation (Training Phase)

Run the official Task 1 & 3 evaluation script on training predictions.

In [9]:
# Run official evaluation on training results
from pathlib import Path
if results and len(results) > 0:
    print("=" * 80)
    print("OFFICIAL EVALUATION — TRAINING (Task 1 & 3)")
    print("=" * 80)

    reference_file = os.path.join(output_dir, 'reference.json')
    predictions_file = os.path.join(output_dir, 'predictions_official.json')
    results_file = os.path.join(output_dir, 'official_evaluation_results.json')

    # Build reference file (ground truth)
    print("\n[1] Building reference file...")
    reference_data = []
    for item in train_data:
        reference_data.append({
            'id': item['id'],
            'validity': item.get('validity', False),
            'plausibility': item.get('plausibility', False)
        })
    with open(reference_file, 'w', encoding='utf-8') as f:
        json.dump(reference_data, f, indent=2)
    print(f"✓ Reference file: {reference_file}")

    # Build predictions file (convert to boolean format)
    print("\n[2] Building predictions file...")
    predictions_official = []
    for r in results:
        validity = (r.get('prediction') == 'VALID')
        predictions_official.append({'id': r['id'], 'validity': validity})
    with open(predictions_file, 'w', encoding='utf-8') as f:
        json.dump(predictions_official, f, indent=2)
    print(f"✓ Predictions file: {predictions_file}")

    # Run official evaluation
    print("\n[3] Running official evaluation...")
    print("-" * 80)
    _eval_kit = Path.cwd().parent / "evaluation_kit" / "task 1 & 3"
    if _eval_kit.exists() and str(_eval_kit) not in sys.path:
        sys.path.insert(0, str(_eval_kit))
    from evaluation_script import run_full_scoring
    run_full_scoring(reference_file, predictions_file, results_file)

    if os.path.exists(results_file):
        print("\n" + "=" * 80)
        print("OFFICIAL EVALUATION RESULTS (TRAINING)")
        print("=" * 80)
        with open(results_file, 'r') as f:
            eval_results = json.load(f)
        print(f"\nModel: {MODEL_ID}")
        print(f"Examples: {len(train_data)}")
        print(f"\nMetrics:")
        print(f"  Accuracy:        {eval_results.get('accuracy', 0):.2f}%")
        print(f"  Content Effect:  {eval_results.get('content_effect', 0):.2f}%")
        print(f"  Combined Score:  {eval_results.get('combined_score', 0):.2f}%")
        print("\n" + "=" * 80)
else:
    print("⏭ Skipping official evaluation (no training results)")

OFFICIAL EVALUATION — TRAINING (Task 1 & 3)

[1] Building reference file...
✓ Reference file: ../semeval_results\incremental_train_20260301_102956\reference.json

[2] Building predictions file...
✓ Predictions file: ../semeval_results\incremental_train_20260301_102956\predictions_official.json

[3] Running official evaluation...
--------------------------------------------------------------------------------
Scoring complete. Results written to ../semeval_results\incremental_train_20260301_102956\official_evaluation_results.json

OFFICIAL EVALUATION RESULTS (TRAINING)

Model: qwen.qwen3-32b-v1:0
Examples: 200

Metrics:
  Accuracy:        93.00%
  Content Effect:  0.94%
  Combined Score:  55.95%



## 8. View Enhanced Prompt Evolution

In [10]:
# Show prompt evolution
prompts_dir = os.path.join(output_dir, "prompts")

print("PROMPT EVOLUTION")
print("=" * 60)

for filename in sorted(os.listdir(prompts_dir)):
    filepath = os.path.join(prompts_dir, filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    print(f"{filename}: {len(content)} chars")

print("\n" + "=" * 60)

PROMPT EVOLUTION
prompt_final.txt: 24585 chars
prompt_v0.txt: 15878 chars
prompt_v1.txt: 18358 chars
prompt_v2.txt: 20193 chars
prompt_v3.txt: 22868 chars
prompt_v4.txt: 24585 chars



In [11]:
# View final enhanced prompt (first 3000 chars)
if final_prompt_path and os.path.exists(final_prompt_path):
    print("FINAL ENHANCED PROMPT PREVIEW")
    print("=" * 60)
    
    with open(final_prompt_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    print(content[:3000])
    if len(content) > 3000:
        print(f"\n... [{len(content) - 3000} more characters]")
    print("\n" + "=" * 60)

FINAL ENHANCED PROMPT PREVIEW
You are a formal logic expert. Your task is to extract the logical structure from syllogisms.

## ⚠️ CRITICAL: Distinguishing I-type from O-type

**This is the most common error. Pay VERY close attention!**

| Type | Pattern | Key Indicator | Example |
|------|---------|---------------|---------|
| **I** | "Some S are P" | NO negation | "Some birds are pets" |
| **O** | "Some S are NOT P" | HAS negation | "Some birds are **not** pets" |

### O-type Indicators (NEGATIVE particular) - Look for these words:
- "Some X are **not** Y"
- "There are X that are **not** Y"
- "**Not all** X are Y"
- "**Not every** X is Y"
- "A portion of X are **not** Y"
- "At least one X is **not** Y"
- "Some X **fail to be** Y"
- "There exist X which are **not** Y"
- "There are some X that are **not** Y"
- "There are a number of X who are **not** Y"

### I-type Indicators (AFFIRMATIVE particular) - NO negation present:
- "Some X are Y" (no negation!)
- "There are X that are Y"
- "A

## 9. TEST PHASE with Final Enhanced Prompt

## 9.1 Official Evaluation (Test Phase)

Run the **next cell** to run the test phase and official Task 1 & 3 evaluation (Accuracy, Content Effect, Combined Score).

In [12]:
# (Official evaluation runs at the end of the next cell.)
pass

In [13]:
test_results = None
if NUM_TEST == 0:
    print("⏭ Skipping test phase (NUM_TEST = 0)")
else:
    # Load test data
    with open(TEST_DATA_PATH, 'r', encoding='utf-8') as f:
        test_data = json.load(f)
    
    if NUM_TEST > 0:
        test_data = test_data[:NUM_TEST]
    
    print(f"Loaded {len(test_data)} test examples")
    
    # Use final enhanced prompt
    if final_prompt_path and os.path.exists(final_prompt_path):
        print(f"\n✓ Using FINAL ENHANCED prompt from incremental meta-learning")
        prompt_to_use = final_prompt_path
    else:
        print(f"\n→ Using BASE prompt (no meta-learning)")
        prompt_to_use = BASE_PROMPT_PATH
    
    # Initialize test pipeline
    test_pipeline = NeuroSymbolicPipeline(
        bedrock_client=client,
        prompt_path=prompt_to_use,
        results_dir=RESULTS_DIR,
        run_name="test_enhanced",
        **PIPELINE_CONFIG
    )
    
    # Run test
    test_results = test_pipeline.run(test_data, verbose=True)
    
    # Save predictions
    test_pipeline.save_predictions(test_results)
    print("\n✓ Predictions saved!")

    # Official evaluation on test (Task 1 & 3)
    print("\n" + "=" * 80)
    print("OFFICIAL EVALUATION — TEST (Task 1 & 3)")
    print("=" * 80)
    test_output_dir = test_pipeline.results_dir
    reference_file = os.path.join(test_output_dir, 'reference.json')
    predictions_file = os.path.join(test_output_dir, 'predictions_official.json')
    results_file = os.path.join(test_output_dir, 'official_evaluation_results.json')
    print("\n[1] Building reference file...")
    reference_data = [{'id': item['id'], 'validity': item.get('validity', False), 'plausibility': item.get('plausibility', False)} for item in test_data]
    with open(reference_file, 'w', encoding='utf-8') as f:
        json.dump(reference_data, f, indent=2)
    print(f"✓ Reference file: {reference_file}")
    print("\n[2] Building predictions file...")
    predictions_official = [{'id': r['id'], 'validity': (r.get('prediction') == 'VALID')} for r in test_results]
    with open(predictions_file, 'w', encoding='utf-8') as f:
        json.dump(predictions_official, f, indent=2)
    print(f"✓ Predictions file: {predictions_file}")
    print("\n[3] Running official evaluation...")
    print("-" * 80)
    _eval_kit = Path.cwd().parent / "evaluation_kit" / "task 1 & 3"
    if _eval_kit.exists() and str(_eval_kit) not in sys.path:
        sys.path.insert(0, str(_eval_kit))
    from evaluation_script import run_full_scoring
    run_full_scoring(reference_file, predictions_file, results_file)
    if os.path.exists(results_file):
        print("\n" + "=" * 80)
        print("OFFICIAL EVALUATION RESULTS (TEST)")
        print("=" * 80)
        with open(results_file, 'r') as f:
            eval_results = json.load(f)
        print(f"\nModel: {MODEL_ID}")
        print(f"Examples: {len(test_data)}")
        print(f"\nMetrics:")
        print(f"  Accuracy:        {eval_results.get('accuracy', 0):.2f}%")
        print(f"  Content Effect:  {eval_results.get('content_effect', 0):.2f}%")
        print(f"  Combined Score:  {eval_results.get('combined_score', 0):.2f}%")
        print("\n" + "=" * 80)

Loaded 191 test examples

✓ Using FINAL ENHANCED prompt from incremental meta-learning
  System prompt saved to: ../semeval_results\neurosymbolic_test_enhanced_20260301_112012\system_prompt.txt
  Results dir: ../semeval_results\neurosymbolic_test_enhanced_20260301_112012
✓ NeuroSymbolic Pipeline initialized
  Extractor: Original (LLM figure)
  Use reflexion: True
  Use self-consistency: True
    Samples: 4, Temperatures: [0.0, 0.3, 0.5, 0.7]
  Verify figure: True
  Use fallback: True
    Fallback self-consistency: False
  Results dir: ../semeval_results\neurosymbolic_test_enhanced_20260301_112012


Processing: 100%|██████████| 191/191 [52:16<00:00, 16.42s/it] 


✓ Summary saved to: ../semeval_results\neurosymbolic_test_enhanced_20260301_112012
  Methods: 171 symbolic, 20 fallback, 0 failed

  ┌─────────────────────────────────────┐
  │  COMBINED SCORE:    31.97           │
  │  Accuracy:          95.29%          │
  │  Content Effect:     6.25           │
  └─────────────────────────────────────┘

  Accuracy breakdown: Symbolic 95.91%, Fallback 90.00%

  Failures logged: 9 (see debug_failures.txt)
✓ Predictions saved to: ../semeval_results\neurosymbolic_test_enhanced_20260301_112012\predictions.json (official format: validity=boolean)

✓ Predictions saved!

OFFICIAL EVALUATION — TEST (Task 1 & 3)

[1] Building reference file...
✓ Reference file: ../semeval_results\neurosymbolic_test_enhanced_20260301_112012\reference.json

[2] Building predictions file...
✓ Predictions file: ../semeval_results\neurosymbolic_test_enhanced_20260301_112012\predictions_official.json

[3] Running official evaluation...
---------------------------------------------